In [1]:
# Pulisce vecchie versioni rotte e installa il pacchetto moderno
!pip uninstall -y numpy faiss-cpu faiss-gpu faiss-gpu-cu12
!pip install "numpy>=2.0.0"
!pip install faiss-gpu-cu12 rank_bm25 sentence-transformers datasets requests

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 107.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 21.3 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


RETRIEVER

In [2]:
import os
import gc
import json
import time
import pickle
import faiss
import torch
import numpy as np
import requests
import torch.nn.functional as F
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# ==========================================
# 1. RAMO KB (Inizializzatore e Retriever)
# ==========================================
class KBBranchInitializer:
    def __init__(self, shared_embedder, cache_dir="/content/drive/MyDrive/MedFactCheck_Cache/"):
        self.cache_dir = cache_dir
        self.embedder = shared_embedder
        os.makedirs(self.cache_dir, exist_ok=True)

        self.texts_paths = [os.path.join(self.cache_dir, f"scifact_texts_v{i}.json") for i in range(1, 4)]
        self.bm25_paths = [os.path.join(self.cache_dir, f"bm25_index_v{i}.pkl") for i in range(1, 4)]
        self.faiss_paths = [os.path.join(self.cache_dir, f"faiss_index_v{i}.index") for i in range(1, 4)]

        self.doc_texts, self.bm25, self.faiss_index = None, None, None
        self._initialize_all()

    def _initialize_all(self):
        # 1. TESTI (Download Bulletproof)
        self.doc_texts = self._load_with_fallback(self.texts_paths, json.load, "r")
        if not self.doc_texts:
            print("-> [KB] Download diretto SciFact (URL Stabile MTEB - Anti-Crash)...")
            url = "https://huggingface.co/datasets/mteb/scifact/resolve/main/corpus.jsonl"
            resp = requests.get(url, timeout=30)

            self.doc_texts = []
            for line in resp.text.strip().split('\n'):
                if line.strip():
                    doc = json.loads(line)
                    # MTEB usa 'text' invece di 'abstract'
                    testo_unito = f"{doc.get('title', '')} {doc.get('text', '')}"
                    self.doc_texts.append(testo_unito)

            self._save_triple_backup(self.doc_texts, self.texts_paths, lambda d, f: json.dump(d, f), "w")

        # 2. BM-25
        self.bm25 = self._load_with_fallback(self.bm25_paths, pickle.load, "rb")
        if not self.bm25:
            print("-> [KB] Creazione indice BM25...")
            self.bm25 = BM25Okapi([doc.lower().split() for doc in self.doc_texts])
            self._save_triple_backup(self.bm25, self.bm25_paths, lambda d, f: pickle.dump(d, f), "wb")

        # 3. FAISS (PQ)
        self.faiss_index = self._load_faiss_with_fallback()
        if not self.faiss_index:
            print("-> [KB] Addestramento FAISS PQ in corso...")
            embeddings = self.embedder.encode(self.doc_texts, convert_to_numpy=True, show_progress_bar=True)

            embeddings = embeddings.astype('float32')

            faiss.normalize_L2(embeddings)
            d, m, nbits = embeddings.shape[1], 96, 8
            self.faiss_index = faiss.IndexPQ(d, m, nbits, faiss.METRIC_INNER_PRODUCT)
            self.faiss_index.train(embeddings)
            self.faiss_index.add(embeddings)
            for path in self.faiss_paths:
                faiss.write_index(self.faiss_index, path)
            torch.cuda.empty_cache()

    def _load_with_fallback(self, paths, load_func, mode):
        for path in paths:
            if os.path.exists(path):
                try:
                    with open(path, mode) as f: return load_func(f)
                except Exception: pass
        return None

    def _load_faiss_with_fallback(self):
        for path in self.faiss_paths:
            if os.path.exists(path):
                try: return faiss.read_index(path)
                except Exception: pass
        return None

    def _save_triple_backup(self, data, paths, save_func, mode):
        for path in paths:
            with open(path, mode) as f: save_func(data, f)

class KBRetrieverNode:
    def __init__(self, kb_data, shared_embedder):
        self.doc_texts = kb_data.doc_texts
        self.bm25 = kb_data.bm25
        self.faiss_index = kb_data.faiss_index
        self.embedder = shared_embedder

    def _apply_hard_truncation(self, text: str, max_chars: int = 500) -> str:
        if len(text) <= max_chars: return text
        return text[:max_chars].rsplit(' ', 1)[0] + "..."

    def search_bm25(self, query: str, top_k: int) -> list:
        scores = self.bm25.get_scores(query.lower().split())
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [{"text": self._apply_hard_truncation(self.doc_texts[idx]),
                 "source": "KB (SciFact - BM25)",
                 "score": round(float(scores[idx]), 4)}
                for idx in top_indices if scores[idx] > 0]

    def search_faiss_pq(self, query: str, top_k: int) -> list:
        query_vector = self.embedder.encode([query], convert_to_numpy=True)

        # FIX: Convertiamo il vettore della query in FP32 per FAISS
        query_vector = query_vector.astype('float32')

        faiss.normalize_L2(query_vector)
        distances, indices = self.faiss_index.search(query_vector, top_k)
        return [{"text": self._apply_hard_truncation(self.doc_texts[idx]),
                 "source": "KB (SciFact - FAISS)",
                 "score": round(float(dist), 4)}
                for dist, idx in zip(distances[0], indices[0]) if idx != -1]


# ==========================================
# 2. RAMO LIT (Europe PMC Massivo)
# ==========================================
class LitBulkRetrieverNode:
    def __init__(self, shared_embedder):
        self.embedder = shared_embedder
        self.pmc_api_url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
        self.mock_db = {}

    def _fetch_top_papers_metadata(self, query: str, limit: int = 50) -> list:
        params = {"query": f'({query}) AND OPEN_ACCESS:y', "format": "json", "resultType": "lite", "pageSize": limit}
        try:
            res = requests.get(self.pmc_api_url, params=params, timeout=10)
            return [{"id": r.get("pmcid") or r.get("pmid"), "title": r.get("title", "")}
                    for r in res.json().get("resultList", {}).get("result", []) if r.get("pmcid") or r.get("pmid")]
        except Exception: return []

    def _fetch_full_text_from_api(self, pmcid: str) -> str:
        params = {"query": pmcid, "format": "json", "resultType": "core"}
        try:
            res = requests.get(self.pmc_api_url, params=params, timeout=5)
            if res.status_code != 200: return ""
            result_list = res.json().get("resultList", {}).get("result", [])
            return result_list[0].get("abstractText", "") if result_list else ""
        except Exception: return ""

    def retrieve_and_rerank_massive(self, claim: str, top_k_bm25: int = 100, top_n_biobert: int = 20) -> list:
        metadata_list = self._fetch_top_papers_metadata(claim, limit=50)
        if not metadata_list: return []

        target_ids = [m["id"] for m in metadata_list]
        cached_papers = {id_p: self.mock_db[id_p] for id_p in target_ids if id_p in self.mock_db}

        final_papers = []
        for meta in metadata_list:
            pid = meta["id"]
            if pid in cached_papers:
                final_papers.append(cached_papers[pid])
            else:
                full_text = self._fetch_full_text_from_api(pid)
                if full_text:
                    paper_doc = {"id": pid, "title": meta["title"], "text": full_text}
                    final_papers.append(paper_doc)
                    self.mock_db[pid] = paper_doc
                time.sleep(0.1) # Pausa di sicurezza

        all_chunks = []
        for p in final_papers:
            words = f"{p['title']}. {p['text']}".split()
            for i in range(0, len(words), 120):
                all_chunks.append({"text": " ".join(words[i:i + 150]), "source": f"PMC ID: {p['id']}"})

        if not all_chunks: return []

        bm25_scores = BM25Okapi([c["text"].lower().split() for c in all_chunks]).get_scores(claim.lower().split())
        candidate_chunks = [all_chunks[i] for i in np.argsort(bm25_scores)[::-1][:min(top_k_bm25, len(all_chunks))] if bm25_scores[i] > 0]

        if not candidate_chunks: return []

        with torch.no_grad():
            claim_emb = self.embedder.encode(claim, convert_to_tensor=True)
            chunk_embs = self.embedder.encode([c["text"] for c in candidate_chunks], convert_to_tensor=True)
            similarities = F.cosine_similarity(claim_emb.unsqueeze(0), chunk_embs)

        top_indices = torch.argsort(similarities, descending=True)[:min(top_n_biobert, len(candidate_chunks))]
        res = [{"text": candidate_chunks[i.item()]["text"], "source": candidate_chunks[i.item()]["source"], "score": round(float(similarities[i.item()]), 4)} for i in top_indices]

        torch.cuda.empty_cache()
        return res


# ==========================================
# 3. L'ORCHESTRATORE GLOBALE
# ==========================================
class MedFactCheckRetriever:
    def __init__(self):
        print(f"[*] Inizializzazione Globale Sistema RAG su: {DEVICE}")
        print("[*] Allocazione S-PubMedBert (FP16) in VRAM...")
        self.embedder = SentenceTransformer('pritamdeka/S-PubMedBert-MS-MARCO', model_kwargs={"torch_dtype": torch.float16}, device=DEVICE)

        print("[*] Sincronizzazione Ramo KB (SciFact)...")
        kb_data = KBBranchInitializer(shared_embedder=self.embedder)
        self.kb_node = KBRetrieverNode(kb_data, shared_embedder=self.embedder)

        print("[*] Sincronizzazione Ramo LIT (Europe PMC)...")
        self.lit_node = LitBulkRetrieverNode(shared_embedder=self.embedder)

        print("\n✅ SISTEMA DI RETRIEVAL PRONTO E ALLINEATO.")

    def retrieve(self, claim: str, routes: list) -> list:
        final_evidence = []
        if routes == ["kb"]:
            print(f" -> [ROTTA KB ESATTA] Esecuzione BM-25 su SciFact...")
            final_evidence.extend(self.kb_node.search_bm25(claim, top_k=3))
        elif "lit" in routes:
            print(f" -> [ROTTA COMPLESSA] 1. FAISS su SciFact...")
            final_evidence.extend(self.kb_node.search_faiss_pq(claim, top_k=3))
            print(f" -> [ROTTA COMPLESSA] 2. Ricerca Massiva su Europe PMC...")
            final_evidence.extend(self.lit_node.retrieve_and_rerank_massive(claim))
        return final_evidence




# Creiamo l'istanza globale. L'embedder entra in VRAM e ci resta.
rag_system = MedFactCheckRetriever()

[*] Inizializzazione Globale Sistema RAG su: cuda:0
[*] Allocazione S-PubMedBert (FP16) in VRAM...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[*] Sincronizzazione Ramo KB (SciFact)...
[*] Sincronizzazione Ramo LIT (Europe PMC)...

✅ SISTEMA DI RETRIEVAL PRONTO E ALLINEATO.


TEST KB

In [3]:
#claim_semplice = "L'ibuprofene è un antinfiammatorio non steroideo."
claim_semplice = "Ibuprofen is a nonsteroidal anti-inflammatory drug."
rotte = ["kb"]

print("\n--- TEST 1: DEFINIZIONE DA KB ---")
risultati_kb = rag_system.retrieve(claim_semplice, rotte)

print(f"\nTrovate {len(risultati_kb)} evidenze:")
for i, ev in enumerate(risultati_kb):
    print(f"[{i+1}] Score: {ev['score']} | {ev['source']} \n    {ev['text'][:150]}...\n")


--- TEST 1: DEFINIZIONE DA KB ---
 -> [ROTTA KB ESATTA] Esecuzione BM-25 su SciFact...

Trovate 3 evidenze:
[1] Score: 26.7195 | KB (SciFact - BM25) 
    Variability in risk of gastrointestinal complications with individual non-steroidal anti-inflammatory drugs: results of a collaborative meta-analysis....

[2] Score: 20.2736 | KB (SciFact - BM25) 
    Ibuprofen for the treatment of patent ductus arteriosus in preterm or low birth weight (or both) infants. BACKGROUND Indomethacin is used as standard ...

[3] Score: 19.3257 | KB (SciFact - BM25) 
    Coxibs and other nonsteroidal anti-inflammatory drugs in animal models of cancer chemoprevention. Coxibs, including celecoxib, and other nonsteroidal ...



TEST KB + LIT

In [4]:
# Usa l'inglese per Europe PMC!
claim_complesso = "ibuprofen long term use gastric ulcer"
rotte = ["kb", "lit"]

print("\n--- TEST 2: RICERCA IBRIDA MASSIVA ---")
risultati_ibridi = rag_system.retrieve(claim_complesso, rotte)

print(f"\nTrovate {len(risultati_ibridi)} evidenze totali da analizzare con Qwen:")
for i, ev in enumerate(risultati_ibridi):
    print(f"[{i+1}] Score: {ev['score']} | {ev['source']} \n    {ev['text'][:150]}...\n")


--- TEST 2: RICERCA IBRIDA MASSIVA ---
 -> [ROTTA COMPLESSA] 1. FAISS su SciFact...
 -> [ROTTA COMPLESSA] 2. Ricerca Massiva su Europe PMC...

Trovate 23 evidenze totali da analizzare con Qwen:
[1] Score: 0.9019 | KB (SciFact - FAISS) 
    Ibuprofen for the treatment of patent ductus arteriosus in preterm or low birth weight (or both) infants. BACKGROUND Indomethacin is used as standard ...

[2] Score: 0.8975 | KB (SciFact - FAISS) 
    Variability in risk of gastrointestinal complications with individual non-steroidal anti-inflammatory drugs: results of a collaborative meta-analysis....

[3] Score: 0.8916 | KB (SciFact - FAISS) 
    Platelet glycoprotein IIb/IIIa inhibitors reduce mortality in diabetic patients with non-ST-segment-elevation acute coronary syndromes. BACKGROUND Dia...

[4] Score: 0.9307 | PMC ID: PMC12015679 
    Over-the-Counter Ibuprofen-Induced Pre-Pyloric Gastric Perforation in a 28-Month-Old Child: A Rare Pediatric Case.. Gastric perforation is a rare but ...

[5

TEST SUITE

In [5]:
import time

# Assicuriamoci che l'oggetto rag_system esista già in memoria dalle celle precedenti
# rag_system = MedFactCheckRetriever() # (Non decommentare se hai già eseguito la Cella 3)

# ==========================================
# DEFINIZIONE DELLA SUITE DI TEST
# ==========================================
test_suite = [
    # --- RAMO KB (Scienza di Base & Meccanismi) ---
    {
        "id": "KB-1-BIO",
        "route": ["kb"],
        "claim": "Caspase-3 activation is a critical step in the execution phase of cellular apoptosis.",
        "desc": "Biologia Molecolare e Apoptosi"
    },
    {
        "id": "KB-2-PHARMA",
        "route": ["kb"],
        "claim": "COX-2 selective inhibitors reduce prostaglandin synthesis while sparing the gastric mucosa compared to non-selective NSAIDs.",
        "desc": "Meccanismo d'Azione Farmacologico"
    },
    {
        "id": "KB-3-VIRO",
        "route": ["kb"],
        "claim": "The ACE2 receptor is the primary entry point for the SARS-CoV-2 virus into human host cells.",
        "desc": "Virologia e Recettori"
    },

    # --- RAMO IBRIDO (Trial Clinici, Linee Guida & Interazioni) ---
    {
        "id": "LIT-1-INTERACT",
        "route": ["kb", "lit"],
        "claim": "Concomitant use of Selective Serotonin Reuptake Inhibitors (SSRIs) and NSAIDs significantly increases the risk of upper gastrointestinal bleeding.",
        "desc": "Interazione Farmacologica (Farmacovigilanza)"
    },
    {
        "id": "LIT-2-TRIAL",
        "route": ["kb", "lit"],
        "claim": "SGLT2 inhibitors, such as empagliflozin, significantly reduce the risk of hospitalization for heart failure in patients with type 2 diabetes.",
        "desc": "Trial Clinici e Nuovi Standard of Care"
    },
    {
        "id": "LIT-3-DEBATE",
        "route": ["kb", "lit"],
        "claim": "High-dose intravenous Vitamin C therapy significantly reduces mortality in late-stage septic shock patients.",
        "desc": "Efficacia Terapeutica in Terapie Dibattute"
    }
]

# ==========================================
# ESECUZIONE BATCH
# ==========================================
print("🚀 AVVIO STRESS TEST DEL RETRIEVER 🚀\n" + "="*50)

for test in test_suite:
    print(f"\n🧪 TEST [{test['id']}] - {test['desc']}")
    print(f"📌 Claim: '{test['claim']}'")
    print(f"🛣️ Rotta: {test['route']}")

    start_time = time.time()

    # Chiamata al nostro motore RAG
    risultati = rag_system.retrieve(test['claim'], test['route'])

    elapsed = time.time() - start_time

    print(f"⏱️ Tempo di estrazione: {elapsed:.2f} secondi")
    print(f"📊 Evidenze trovate: {len(risultati)}")

    # Stampiamo solo le prime 2 evidenze per non intasare l'output,
    # ma visualizzando i punteggi e la fonte.
    for i, ev in enumerate(risultati):
        testo_pulito = ev['text'].replace('\n', ' ')
        print(f"  [{i+1}] Score: {ev['score']} | {ev['source']}")
        print(f"      {testo_pulito[:180]}...")

    print("-" * 50)

🚀 AVVIO STRESS TEST DEL RETRIEVER 🚀

🧪 TEST [KB-1-BIO] - Biologia Molecolare e Apoptosi
📌 Claim: 'Caspase-3 activation is a critical step in the execution phase of cellular apoptosis.'
🛣️ Rotta: ['kb']
 -> [ROTTA KB ESATTA] Esecuzione BM-25 su SciFact...
⏱️ Tempo di estrazione: 0.03 secondi
📊 Evidenze trovate: 3
  [1] Score: 34.3845 | KB (SciFact - BM25)
      Identification of a novel p53 functional domain that is necessary for mediating apoptosis. The ability of p53 to induce apoptosis requires its sequence-specific DNA binding activit...
  [2] Score: 33.6179 | KB (SciFact - BM25)
      Metformin reverses established lung fibrosis in a bleomycin model Fibrosis is a pathological result of a dysfunctional repair response to tissue injury and occurs in a number of or...
  [3] Score: 31.1114 | KB (SciFact - BM25)
      Mechanisms of pancreatic beta-cell death in type 1 and type 2 diabetes: many differences, few similarities. Type 1 and type 2 diabetes are characterized by progressive bet